# ResNet classifier

## Importations

In [37]:
import os
import time
import pickle
import multiprocessing
from collections import Counter
 
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.effects
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
 
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
 
import tensorflow as tf
import keras
from keras import layers, models, callbacks, regularizers
from keras.utils import to_categorical
 
import wandb
from wandb.integration.keras import WandbMetricsLogger
 
from classification.datasets import Dataset
from classification.utils.audio_student import AudioUtil, Feature_vector_DS
 
np.random.seed(42)
tf.random.set_seed(42)

## Hardware

In [38]:
def setup_hardware():
    gpus = tf.config.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
 
    n_cores = multiprocessing.cpu_count()
    tf.config.threading.set_intra_op_parallelism_threads(n_cores)
    tf.config.threading.set_inter_op_parallelism_threads(4)
 
    print(f"TF {tf.__version__} | Keras {keras.__version__}")
    print(f"GPU Metal : {'OK — ' + gpus[0].name if gpus else 'not detected'}")
    print(f"CPU cores : {n_cores}")
 
 
setup_hardware()

TF 2.18.1 | Keras 3.12.1
GPU Metal : OK — /physical_device:GPU:0
CPU cores : 16


## Config

In [ ]:
class Config:
    # -- DSP ------------------------------------------------------------------
    SAMPLE_RATE = 11025    # Hz — must match MCU firmware, so 10500 ? i don't remember
    N_MEL       = 64       # mel bands         | try: lower
    N_FFT       = 512      # FFT window size   | try: 512, 1024
    HOP_LENGTH  = 128      # FFT hop           | rule: N_FFT // 4
    DURATION_MS = 1000     # analysis window   | try: 750, 1000, 1500
 
    # -- Augmentation ---------------------------------------------------------
    ENABLE_TIME_SHIFT   = True
    ENABLE_PITCH_SHIFT  = True
    ENABLE_TIME_STRETCH = True
    ENABLE_NOISE        = True
    ENABLE_SPEC_MASKING = True
 
    PITCH_SHIFT_SEMITONES = (-3, 3)      # try: (-2,2), (-4,4)
    TIME_STRETCH_RANGE    = (0.85, 1.15) # try: (0.9,1.1), (0.8,1.2)
    NOISE_SIGMA           = 0.04         # try: 0.02, 0.06
 
    # -- Architecture ---------------------------------------------------------
    # Current results: train=100%, val=98%, test=86% -> domain gap, not overfitting.
    # Priority: more real data > regularisation changes.
    DROPOUT_RATE    = 0.3   # try: 0.3 (val generalises well already)
    L2_REG          = 1e-4  # try: 5e-5, 2e-4
    LABEL_SMOOTHING = 0.15   # try: 0.05, 0.15
    MIXUP_ALPHA     = 0.4   # try: 0.4 to help fire/fireworks boundary
 
    # -- Training -------------------------------------------------------------
    BATCH_SIZE    = 256     # GPU-optimal; try 128 if OOM
    EPOCHS        = 300
    LEARNING_RATE = 5e-4   # try: 3e-4, 5e-4
    EARLY_STOPPING_PATIENCE = 60
    TTA_STEPS = 5           # test-time augmentation passes
 
    # -- LR schedule ----------------------------------------------------------
    # CosineDecayRestarts: LR completes a full cycle even with early stopping.
    # m_mul<1 ensures each restart begins lower than the previous.
    COSINE_RESTART_EPOCHS = 45   # try: 40, 50
    COSINE_M_MUL          = 0.65 # try: 0.7, 0.6
 
    # -- Data paths -----------------------------------------------------------
    # REAL_DATA_DIR contains subfolders per class, each with subfolders:
    #   training/  — used for train pool
    #   test/      — held-out evaluation set, never touched during training
    REAL_DATA_DIR        = "../mcu/hands_on_audio_acquisition/audio_files"
    REAL_TRAIN_SUBFOLDERS = ["training"]  # extend if you add new recording sessions
    SYNTH_AUG_PASSES     = 0    # augmentation passes on synthetic data; 0 = disabled
    REAL_AUG_PASSES      = 60   # augmentation passes on real recordings
    TRAIN_SPLIT_RATIO    = 0.8  # fraction of pool assigned to train
 
    # -- Output ---------------------------------------------------------------
    MODEL_DIR   = "./data/models/models_resnet/lunS8_10:50"
    RANDOM_SEED = 42
 
 
config = Config()

## Augmentation

In [40]:
class AudioAugmentation:
 
    @staticmethod
    def time_shift(signal: np.ndarray, shift_max: float = 0.2) -> np.ndarray:
        shift = int(np.random.uniform(-shift_max, shift_max) * len(signal))
        out = np.zeros_like(signal)
        if shift > 0:
            out[shift:] = signal[:-shift]
        elif shift < 0:
            out[:shift] = signal[-shift:]
        else:
            out = signal.copy()
        return out
 
    @staticmethod
    def pitch_shift(signal: np.ndarray, sr: int, semitone_range: tuple) -> np.ndarray:
        n_steps = np.random.uniform(*semitone_range)
        return librosa.effects.pitch_shift(signal, sr=sr, n_steps=n_steps)
 
    @staticmethod
    def time_stretch(signal: np.ndarray, rate_range: tuple) -> np.ndarray:
        rate = np.random.uniform(*rate_range)
        stretched = librosa.effects.time_stretch(signal, rate=rate)
        if len(stretched) > len(signal):
            return stretched[:len(signal)]
        return np.pad(stretched, (0, len(signal) - len(stretched)))
 
    @staticmethod
    def apply(audio: tuple, cfg: Config, skip_noise: bool = False) -> tuple:
        sig, sr = audio
        if cfg.ENABLE_TIME_SHIFT and np.random.random() > 0.5:
            sig = AudioAugmentation.time_shift(sig)
        if cfg.ENABLE_PITCH_SHIFT and np.random.random() > 0.5:
            sig = AudioAugmentation.pitch_shift(sig, sr, cfg.PITCH_SHIFT_SEMITONES)
        if cfg.ENABLE_TIME_STRETCH and np.random.random() > 0.5:
            sig = AudioAugmentation.time_stretch(sig, cfg.TIME_STRETCH_RANGE)
        if not skip_noise and cfg.ENABLE_NOISE and np.random.random() > 0.5:
            sig, sr = AudioUtil.add_noise((sig, sr), sigma=cfg.NOISE_SIGMA)
        return sig, sr

## Synthetic dataset wrapper (bridges legacy Feature_vector_DS)

In [41]:
class SynthDataset(Feature_vector_DS):
 
    def __init__(self, *args, augment: bool = False, cfg: Config = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.augment = augment
        self.cfg = cfg or Config()
 
    def get_audiosignal(self, idx):
        aud = AudioUtil.open(self.dataset[idx])
        aud = AudioUtil.resample(aud, self.sr)
        if self.augment:
            aud = AudioAugmentation.apply(aud, self.cfg)
        if self.data_aug:
            if "add_bg" in self.data_aug:
                aud = AudioUtil.add_bg(aud, self.dataset, num_sources=1,
                                       max_ms=self.duration, amplitude_limit=0.1)
            if "noise"   in self.data_aug: aud = AudioUtil.add_noise(aud, sigma=0.05)
            if "echo"    in self.data_aug: aud = AudioUtil.echo(aud)
            if "scaling" in self.data_aug: aud = AudioUtil.scaling(aud, scaling_limit=5)
        sig, sr = aud
        return sig / (np.max(np.abs(sig)) + 1e-8), sr
 
    def __getitem__(self, idx):
        aud = self.get_audiosignal(idx)
        spec = AudioUtil.melspectrogram(aud, Nmel=self.nmel, Nft=self.Nft)
        if self.augment and self.cfg.ENABLE_SPEC_MASKING and np.random.random() > 0.5:
            spec = AudioUtil.spectro_aug_timefreq_masking(
                spec, max_mask_pct=0.1, n_freq_masks=2, n_time_masks=2)
        return spec
 
    def treat_spec(self, spec):
        n_cols = spec.shape[1]
        if n_cols < self.ncol:
            spec = np.pad(spec, ((0, 0), (0, self.ncol - n_cols)), constant_values=0)
            n_cols = self.ncol
        idxs = np.arange(0, n_cols - self.ncol + 1, self.step, dtype=int)
        if len(idxs) == 0:
            idxs = np.array([0])
        windows = []
        for i in idxs:
            w = spec[:, i:i + self.ncol]
            if w.shape[1] < self.ncol:
                w = np.pad(w, ((0, 0), (0, self.ncol - w.shape[1])), constant_values=0)
            windows.append(w)
        fv = np.array(windows).reshape(len(windows), -1)
        if self.normalize:
            norms = np.linalg.norm(fv, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            fv /= norms
        if self.pca is not None:
            fv = np.array([self.pca.transform([v])[0] for v in fv])
        return fv

## Feature extraction

In [42]:
class MelExtractor:
    """Converts a raw audio file into (N_MEL, n_frames) mel spectrogram windows."""
 
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.n_frames = int(cfg.DURATION_MS / 1000 * cfg.SAMPLE_RATE / cfg.HOP_LENGTH) + 1
        self.step_frames = max(1, self.n_frames // 2)
 
    @property
    def input_shape(self) -> tuple:
        return (self.cfg.N_MEL, self.n_frames, 1)
 
    def _compute_mel(self, signal: np.ndarray, sr: int) -> np.ndarray:
        signal = signal / (np.max(np.abs(signal)) + 1e-8)
        mel = librosa.feature.melspectrogram(
            y=signal, sr=sr,
            n_mels=self.cfg.N_MEL,
            n_fft=self.cfg.N_FFT,
            hop_length=self.cfg.HOP_LENGTH,
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)
        # Per-frequency normalisation (zero mean, unit variance)
        log_mel = (log_mel - log_mel.mean(axis=1, keepdims=True)) / \
                  (log_mel.std(axis=1, keepdims=True) + 1e-8)
        return log_mel
 
    def _sliding_windows(self, log_mel: np.ndarray) -> list:
        T = log_mel.shape[1]
        if T < self.n_frames:
            log_mel = np.pad(log_mel, ((0, 0), (0, self.n_frames - T)))
            T = self.n_frames
        windows = [
            log_mel[:, s:s + self.n_frames]
            for s in range(0, T - self.n_frames + 1, self.step_frames)
        ]
        return windows if windows else [log_mel[:, :self.n_frames]]
 
    def extract_from_audio(self, audio: tuple, augment: bool = False,
                           skip_noise: bool = False) -> list:
        sig, sr = audio
        if augment:
            sig, sr = AudioAugmentation.apply((sig, sr), self.cfg, skip_noise=skip_noise)
        log_mel = self._compute_mel(sig, sr)
        if augment and self.cfg.ENABLE_SPEC_MASKING and np.random.random() > 0.5:
            log_mel = AudioUtil.spectro_aug_timefreq_masking(
                log_mel, max_mask_pct=0.15, n_freq_masks=3, n_time_masks=3)
        return self._sliding_windows(log_mel)
 
    def extract_from_file(self, path: str, augment: bool = False,
                          skip_noise: bool = False) -> list:
        aud = AudioUtil.open(path)
        aud = AudioUtil.resample(aud, self.cfg.SAMPLE_RATE)
        return self.extract_from_audio(aud, augment=augment, skip_noise=skip_noise)

## Data preparation

In [43]:
def _load_wav_dir(folder: str, classnames: list, label_map: dict,
                  extractor: MelExtractor, aug_passes: int = 0,
                  subfolder: str = None, skip_noise: bool = False) -> tuple:
    """Load all .wav files from folder[/subfolder] with optional augmentation."""
    X_list, y_list = [], []
    for cls in classnames:
        path = os.path.join(folder, cls, subfolder) if subfolder else os.path.join(folder, cls)
        if not os.path.exists(path):
            print(f"    SKIP  {path}")
            continue
        files = sorted(f for f in os.listdir(path) if f.endswith(".wav"))
        print(f"    {cls:<15} {len(files):>4} files")
        for fname in files:
            fp = os.path.join(path, fname)
            for w in extractor.extract_from_file(fp, augment=False):
                X_list.append(w)
                y_list.append(label_map[cls])
            for _ in range(aug_passes):
                for w in extractor.extract_from_file(fp, augment=True, skip_noise=skip_noise):
                    X_list.append(w)
                    y_list.append(label_map[cls])
    return np.array(X_list)[..., np.newaxis], np.array(y_list)
 
 
def _synth_dataset_to_array(dataset, classnames: list, extractor: MelExtractor,
                             augment: bool, cfg: Config) -> tuple:
    """Extract feature vectors from the synthetic dataset and reshape to 4D."""
    label_map = {c: i for i, c in enumerate(classnames)}
    n_mel, n_frames = extractor.cfg.N_MEL, extractor.n_frames
    expected = n_mel * n_frames
 
    ds = SynthDataset(
        dataset,
        Nft=cfg.N_FFT, nmel=cfg.N_MEL,
        duration=cfg.DURATION_MS, step=cfg.DURATION_MS // 2,
        augment=augment, cfg=cfg,
    )
    X_flat, y_str = ds.get_feature_vectors()
 
    X_list, y_list = [], []
    for vec, label in zip(X_flat, y_str):
        if len(vec) == expected:
            window = vec.reshape(n_mel, n_frames)
        else:
            window = np.interp(
                np.linspace(0, len(vec) - 1, expected),
                np.arange(len(vec)), vec,
            ).reshape(n_mel, n_frames)
        X_list.append(window)
        y_list.append(label_map[label])
    return np.array(X_list)[..., np.newaxis], np.array(y_list)
 
 
def _print_split_summary(classnames, y_train, y_val, y_test, X_train, X_orig_train):
    SEP, sep = "=" * 70, "-" * 70
    print(f"\n{SEP}\n  DATASET SUMMARY\n{SEP}")
    header = "  " + f"{'Split':<10} {'Total':>8}  " + \
             "  ".join(f"{c[:7]:>8}" for c in classnames)
    print(header)
    print(sep)
    for name, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
        counts = Counter(y)
        row = "  ".join(f"{counts.get(i, 0):>8}" for i in range(len(classnames)))
        print(f"  {name:<10} {len(y):>8}  {row}")
    print(sep)
    pct = 100 * len(X_orig_train) / len(X_train)
    print(f"  Original / augmented in train : {pct:.1f}% / {100 - pct:.1f}%")
    print(SEP)
 
 
def prepare_dataset(synth_dataset, cfg: Config, extractor: MelExtractor,
                    synth_aug_passes: int = None) -> tuple:
    """
    Build train / val / test splits with zero data leakage.
 
    Split strategy:
        1. Load all raw files (real + synthetic, 0 augmentation).
        2. Stratified split on raw pool -> train_raw / val (val stays pure).
        3. Augment train_raw only -> final train set.
        4. Test set is fully isolated from the pool.
    """
    if synth_aug_passes is None:
        synth_aug_passes = cfg.SYNTH_AUG_PASSES
 
    classnames = synth_dataset.list_classes()
    label_map  = {c: i for i, c in enumerate(classnames)}
    real_subfolders = cfg.REAL_TRAIN_SUBFOLDERS
 
    SEP, sep = "=" * 70, "-" * 70
    print(f"\n{SEP}\n  DATASET PREPARATION\n{SEP}")
 
    # -- 1. Real recordings (raw) --------------------------------------------
    real_parts_X, real_parts_y = [], []
    print(f"\n  [1/4] Real recordings  ({cfg.REAL_DATA_DIR})")
    print(f"        Subfolders : {real_subfolders}\n{sep}")
    for subfolder in real_subfolders:
        if not any(os.path.exists(os.path.join(cfg.REAL_DATA_DIR, c, subfolder))
                   for c in classnames):
            print(f"  SKIP  {subfolder}/ not found")
            continue
        X_sub, y_sub = _load_wav_dir(
            cfg.REAL_DATA_DIR, classnames, label_map, extractor,
            aug_passes=0, subfolder=subfolder, skip_noise=True,
        )
        real_parts_X.append(X_sub)
        real_parts_y.append(y_sub)
 
    if real_parts_X:
        X_real = np.concatenate(real_parts_X, axis=0)
        y_real = np.concatenate(real_parts_y, axis=0)
        counts = Counter(y_real)
        print(f"{sep}\n  Real loaded  : {len(X_real)} feature vectors")
        for i, c in enumerate(classnames):
            print(f"    {c:<15} {counts[i]:>5}")
    else:
        X_real = np.empty((0, *extractor.input_shape))
        y_real = np.empty((0,), dtype=int)
 
    # -- 2. Synthetic data (raw) ---------------------------------------------
    print(f"\n  [2/4] Synthetic data (0 augmentation)\n{sep}")
    X_synth, y_synth = _synth_dataset_to_array(
        synth_dataset, classnames, extractor, augment=False, cfg=cfg)
    counts = Counter(y_synth)
    print(f"  Synth loaded : {len(X_synth)} feature vectors")
    for i, c in enumerate(classnames):
        print(f"    {c:<15} {counts[i]:>5}")
 
    # -- 3. Stratified split on raw pool -------------------------------------
    print(f"\n  [3/4] Stratified split "
          f"{int(cfg.TRAIN_SPLIT_RATIO*100)}/{int((1-cfg.TRAIN_SPLIT_RATIO)*100)}\n{sep}")
 
    all_X = [X_synth] + (real_parts_X if real_parts_X else [])
    all_y = [y_synth] + (real_parts_y if real_parts_y else [])
    X_pool = np.concatenate(all_X, axis=0)
    y_pool = np.concatenate(all_y, axis=0)
    assert X_pool.ndim == 4, f"Expected 4D pool, got {X_pool.shape}"
 
    X_train_raw, X_val, y_train_raw, y_val = train_test_split(
        X_pool, y_pool,
        test_size=1 - cfg.TRAIN_SPLIT_RATIO,
        stratify=y_pool,
        random_state=cfg.RANDOM_SEED,
    )
    print(f"  Pool total   : {len(X_pool)} fv  {X_pool.shape[1:]}")
    print(f"  -> Train raw : {len(X_train_raw)}")
    print(f"  -> Val (pure): {len(X_val)}")
 
    # -- 4. Augmentation on train partition only -----------------------------
    print(f"\n  [4/4] Augmentation (train only)\n{sep}")
    aug_X = [X_train_raw]
    aug_y = [y_train_raw]
 
    if synth_aug_passes > 0:
        print(f"  Synthetic : {synth_aug_passes} passes")
        for i in range(synth_aug_passes):
            print(f"    pass {i+1:>3}/{synth_aug_passes}...", end="\r")
            Xp, yp = _synth_dataset_to_array(
                synth_dataset, classnames, extractor, augment=True, cfg=cfg)
            aug_X.append(Xp)
            aug_y.append(yp)
        print(f"  Synthetic : {synth_aug_passes} passes OK" + " " * 20)
    else:
        print(f"  Synthetic : 0 passes (SYNTH_AUG_PASSES=0)")
 
    if cfg.REAL_AUG_PASSES > 0 and real_parts_X:
        print(f"  Real      : {cfg.REAL_AUG_PASSES} passes  {real_subfolders}")
        aug_real_X, aug_real_y = [], []
        for subfolder in real_subfolders:
            for cls in classnames:
                path = os.path.join(cfg.REAL_DATA_DIR, cls, subfolder)
                if not os.path.exists(path):
                    continue
                for fname in sorted(os.listdir(path)):
                    if not fname.endswith(".wav"):
                        continue
                    fp = os.path.join(path, fname)
                    for _ in range(cfg.REAL_AUG_PASSES):
                        for w in extractor.extract_from_file(fp, augment=True, skip_noise=True):
                            aug_real_X.append(w)
                            aug_real_y.append(label_map[cls])
        if aug_real_X:
            X_real_aug = np.array(aug_real_X)[..., np.newaxis]
            y_real_aug = np.array(aug_real_y)
            aug_X.append(X_real_aug)
            aug_y.append(y_real_aug)
            counts = Counter(y_real_aug)
            print(f"  Real aug  : {len(X_real_aug)} fv generated")
            for i, c in enumerate(classnames):
                print(f"    {c:<15} {counts.get(i, 0):>5}")
        else:
            print(f"  WARN  no .wav files found in {real_subfolders}")
    else:
        print(f"  Real      : 0 passes")
 
    X_train = np.concatenate(aug_X, axis=0).astype(np.float32)
    y_train = np.concatenate(aug_y, axis=0)
    X_val   = X_val.astype(np.float32)
 
    # -- Test set (fully isolated) -------------------------------------------
    print(f"\n  Test set  ({cfg.REAL_DATA_DIR}/.../test/)\n{sep}")
    X_test, y_test = _load_wav_dir(
        cfg.REAL_DATA_DIR, classnames, label_map, extractor,
        aug_passes=0, subfolder="test",
    )
    X_test = X_test.astype(np.float32)
 
    _print_split_summary(classnames, y_train, y_val, y_test, X_train, X_train_raw)
    return X_train, X_val, X_test, y_train, y_val, y_test, classnames

## Architecture | ResNet-small audio

In [44]:
def _residual_block(x, filters: int, stride: int = 1, l2: float = 1e-4):
    reg = regularizers.l2(l2)
    shortcut = x
 
    x = layers.Conv2D(filters, 3, strides=stride, padding="same",
                      kernel_regularizer=reg, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding="same",
                      kernel_regularizer=reg, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
 
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride,
                                 kernel_regularizer=reg, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
 
    return layers.ReLU()(layers.Add()([x, shortcut]))
 
 
def build_model(input_shape: tuple, n_classes: int, cfg: Config) -> keras.Model:
    """
    ResNet-small for audio spectrograms.
 
    (N_MEL, T, 1) -> Stem(32) -> ResBlock x5 -> GAP -> Dense -> Softmax
    """
    reg = regularizers.l2(cfg.L2_REG)
    inp = layers.Input(shape=input_shape)
 
    x = layers.Conv2D(32, 5, padding="same", kernel_regularizer=reg, use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPool2D(2, padding="same")(x)
 
    x = _residual_block(x, 64,  stride=1, l2=cfg.L2_REG)
    x = _residual_block(x, 64,  stride=1, l2=cfg.L2_REG)
    x = _residual_block(x, 128, stride=2, l2=cfg.L2_REG)
    x = _residual_block(x, 128, stride=1, l2=cfg.L2_REG)
    x = _residual_block(x, 256, stride=2, l2=cfg.L2_REG)
 
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(cfg.DROPOUT_RATE)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
 
    model = models.Model(inp, out, name="ResNet_Audio")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg.LEARNING_RATE),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=cfg.LABEL_SMOOTHING),
        metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=2)],
    )
    n_params = model.count_params()
    print(f"  Parameters : {n_params:,}  (~{n_params * 4 / 1024:.0f} KB FP32)")
    return model

## MixUp
"C'est contre-intuitif mais extrêmement efficace sur petit dataset." Créer des exemples impossibles (30% chainsaw + 70% fire) et le modèle apprend que la frontière entre les classes est floue. Résultat : il généralise mieux sur des sons "intermédiaires" comme ceux captés sur le micro embarqués

In [45]:
def _mixup(X: np.ndarray, y: np.ndarray, alpha: float) -> tuple:
    if alpha == 0:
        return X, y
    lam = np.random.beta(alpha, alpha)
    perm = np.random.permutation(len(X))
    return lam * X + (1 - lam) * X[perm], lam * y + (1 - lam) * y[perm]
 
 
class MixupGenerator(keras.utils.Sequence):
 
    def __init__(self, X, y, batch_size, n_classes, alpha=0.3, shuffle=True):
        self.X       = X
        self.y_cat   = to_categorical(y, n_classes)
        self.bs      = batch_size
        self.alpha   = alpha
        self.shuffle = shuffle
        self.idx     = np.arange(len(X))
 
    def __len__(self):
        return int(np.ceil(len(self.X) / self.bs))
 
    def __getitem__(self, i):
        batch = self.idx[i * self.bs:(i + 1) * self.bs]
        Xb, yb = _mixup(self.X[batch], self.y_cat[batch], self.alpha)
        return Xb, yb
 
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.idx)

## Callbacks

In [46]:
class GracefulStop(keras.callbacks.Callback):
    """Write ./stop_training.flag from another cell to stop cleanly after the current epoch."""
    FLAG = "./stop_training.flag"
 
    def on_epoch_end(self, epoch, logs=None):
        if os.path.exists(self.FLAG):
            print(f"\n  GracefulStop: flagged at epoch {epoch + 1}")
            os.remove(self.FLAG)
            self.model.stop_training = True
 
 
def _build_callbacks(cfg: Config, steps_per_epoch: int) -> list:
    lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=cfg.LEARNING_RATE,
        first_decay_steps=steps_per_epoch * cfg.COSINE_RESTART_EPOCHS,
        t_mul=1.0,
        m_mul=cfg.COSINE_M_MUL,
        alpha=1e-6,
    )
    # Assign schedule to the already-compiled model's optimiser in train_model.
    return lr_schedule, [
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=cfg.EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ModelCheckpoint(
            os.path.join(cfg.MODEL_DIR, "best_model.keras"),
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        WandbMetricsLogger(log_freq="epoch"),
        GracefulStop(),
    ]

## Training

In [47]:
def train_model(model: keras.Model, X_train, y_train, X_val, y_val,
                cfg: Config, n_classes: int):
    train_gen = MixupGenerator(X_train, y_train, cfg.BATCH_SIZE, n_classes, alpha=cfg.MIXUP_ALPHA)
    y_val_cat = to_categorical(y_val, n_classes)
 
    lr_schedule, cbs = _build_callbacks(cfg, steps_per_epoch=len(train_gen))
    model.optimizer.learning_rate = lr_schedule
 
    print(f"\n  Train : {len(X_train)} samples | {len(train_gen)} batches/epoch")
    print(f"  Val   : {len(X_val)} samples")
 
    history = model.fit(
        train_gen,
        validation_data=(X_val, y_val_cat),
        epochs=cfg.EPOCHS,
        callbacks=cbs,
        verbose=1,
    )
    with open(os.path.join(cfg.MODEL_DIR, "history.pkl"), "wb") as f:
        pickle.dump(history.history, f)
    return history

## Inference

In [48]:
def predict_with_tta(model: keras.Model, X: np.ndarray, n_steps: int) -> np.ndarray:
    """Average predictions over n_steps slightly-shifted copies of X."""
    probs = model.predict(X, verbose=0)
    for _ in range(n_steps - 1):
        X_aug = X.copy()
        shifts = np.random.randint(-3, 4, size=len(X))
        for i, s in enumerate(shifts):
            X_aug[i, :, :, 0] = np.roll(X_aug[i, :, :, 0], s, axis=1)
        probs += model.predict(X_aug, verbose=0)
    return probs / n_steps

## Evaluation

In [49]:
def _plot_confusion_matrices(sets: list, classnames: list, save_path: str):
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    for ax, (title, y_true, y_pred, acc) in zip(axes, sets):
        cm = confusion_matrix(y_true, y_pred)
        im = ax.imshow(cm, cmap="Blues")
        ax.set_title(f"{title}\n{100*acc:.1f}%", fontsize=14, fontweight="bold")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ticks = np.arange(len(classnames))
        ax.set_xticks(ticks); ax.set_xticklabels(classnames, rotation=45, ha="right")
        ax.set_yticks(ticks); ax.set_yticklabels(classnames)
        thresh = cm.max() / 2
        for i in range(len(classnames)):
            for j in range(len(classnames)):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > thresh else "black")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.suptitle("Confusion matrices — Train / Val / Test", fontsize=15,
                 fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
 
 
def _plot_per_class_metrics(classnames, prec, rec, f1, sup):
    x, w = np.arange(len(classnames)), 0.25
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
 
    ax1.bar(x - w, prec, w, label="Precision", color="#2196F3")
    ax1.bar(x,     rec,  w, label="Recall",    color="#4CAF50")
    ax1.bar(x + w, f1,   w, label="F1",        color="#9C27B0")
    ax1.set_xticks(x); ax1.set_xticklabels(classnames, rotation=20, ha="right")
    ax1.set_ylim(0, 1.15)
    ax1.set_title("Precision / Recall / F1 per class", fontweight="bold")
    ax1.legend(); ax1.grid(axis="y", alpha=0.3)
    for i, (p, r, f) in enumerate(zip(prec, rec, f1)):
        ax1.text(i - w, p + 0.02, f"{p:.2f}", ha="center", fontsize=9)
        ax1.text(i,     r + 0.02, f"{r:.2f}", ha="center", fontsize=9)
        ax1.text(i + w, f + 0.02, f"{f:.2f}", ha="center", fontsize=9)
 
    bars = ax2.bar(classnames, sup, color="#FF9800", alpha=0.8)
    ax2.set_title("Support per class (test set)", fontweight="bold")
    ax2.grid(axis="y", alpha=0.3)
    for bar, s in zip(bars, sup):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 str(int(s)), ha="center", fontsize=11)
    plt.tight_layout(); plt.show()
 
 
def evaluate_model(model: keras.Model, X_train, y_train, X_val, y_val,
                   X_test, y_test, classnames: list, cfg: Config) -> dict:
    n_classes = len(classnames)
 
    y_train_pred = np.argmax(model.predict(X_train, verbose=0), axis=1)
    y_val_pred   = np.argmax(model.predict(X_val,   verbose=0), axis=1)
    y_test_probs = predict_with_tta(model, X_test, cfg.TTA_STEPS)
    y_test_pred  = np.argmax(y_test_probs, axis=1)
 
    acc_train = np.mean(y_train_pred == y_train)
    acc_val   = np.mean(y_val_pred   == y_val)
    acc_test  = np.mean(y_test_pred  == y_test)
 
    print(f"\n  Train : {100*acc_train:.2f}%")
    print(f"  Val   : {100*acc_val:.2f}%")
    print(f"  Test  : {100*acc_test:.2f}%  (TTA x{cfg.TTA_STEPS})")
    print(f"  Gap train/test : {100*(acc_train - acc_test):.2f}%")
    if   acc_train - acc_val > 0.15: print("  WARNING  significant overfitting (>15%)")
    elif acc_train - acc_val > 0.08: print("  WARNING  moderate overfitting (>8%)")
    else:                            print("  OK  good train/val balance")
 
    _plot_confusion_matrices([
        ("Training",   y_train, y_train_pred, acc_train),
        ("Validation", y_val,   y_val_pred,   acc_val),
        ("Test (TTA)", y_test,  y_test_pred,  acc_test),
    ], classnames, os.path.join(cfg.MODEL_DIR, "confusion_matrices.png"))
 
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_test, y_test_pred, labels=range(n_classes), average=None)
 
    print(f"\n{'Class':<15} {'Precision':>10} {'Recall':>10} {'F1':>8} {'Support':>10}")
    print("-" * 58)
    for i, c in enumerate(classnames):
        print(f"  {c:<13} {prec[i]:>10.3f} {rec[i]:>10.3f} {f1[i]:>8.3f} {int(sup[i]):>10}")
    print("-" * 58)
    print(f"  {'Macro avg':<13} {prec.mean():>10.3f} {rec.mean():>10.3f} "
          f"{f1.mean():>8.3f} {int(sup.sum()):>10}")
 
    _plot_per_class_metrics(classnames, prec, rec, f1, sup)
 
    for split_name, y_true, y_pred in [
        ("train", y_train, y_train_pred),
        ("val",   y_val,   y_val_pred),
        ("test",  y_test,  y_test_pred),
    ]:
        wandb.log({
            f"confusion/{split_name}": wandb.plot.confusion_matrix(
                y_true=y_true.tolist(), preds=y_pred.tolist(),
                class_names=classnames, title=f"Confusion — {split_name}",
            )
        })
 
    table = wandb.Table(columns=["class", "precision", "recall", "f1", "support"])
    for i, c in enumerate(classnames):
        table.add_data(c, round(float(prec[i]), 3), round(float(rec[i]), 3),
                       round(float(f1[i]), 3), int(sup[i]))
    wandb.log({"metrics_per_class": table})
 
    return dict(train=acc_train, val=acc_val, test=acc_test,
                test_probs=y_test_probs, test_pred=y_test_pred)
 

## Training history plot

In [50]:
def plot_training_history(history, cfg: Config):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metric, title in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
        ax.plot(history.history[metric],           label="Train", lw=2)
        ax.plot(history.history[f"val_{metric}"],  label="Val",   lw=2)
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)
    best = np.argmax(history.history["val_accuracy"])
    axes[0].axvline(best, color="red", linestyle="--", alpha=0.5,
                    label=f"best epoch {best+1}")
    axes[0].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.MODEL_DIR, "training_history.png"), dpi=150)
    plt.show()
    print(f"  Best val accuracy : {100*history.history['val_accuracy'][best]:.2f}%  "
          f"(epoch {best+1})")

## Saving the model

In [51]:
def save_model_and_metadata(model: keras.Model, classnames: list,
                             extractor: MelExtractor, metrics: dict,
                             cfg: Config):
    os.makedirs(cfg.MODEL_DIR, exist_ok=True)
    model_path = os.path.join(cfg.MODEL_DIR, "best_model.keras")
    model.save(model_path)
    model.save_weights(os.path.join(cfg.MODEL_DIR, "best_model.weights.h5"))
 
    meta = dict(
        classnames=classnames,
        input_shape=extractor.input_shape,
        n_mel=cfg.N_MEL,
        n_fft=cfg.N_FFT,
        hop_length=cfg.HOP_LENGTH,
        sample_rate=cfg.SAMPLE_RATE,
        test_accuracy=metrics["test"],
        keras_version=keras.__version__,
    )
    with open(os.path.join(cfg.MODEL_DIR, "model_config.pkl"), "wb") as f:
        pickle.dump(meta, f)
    print(f"  Model saved to {model_path}")

## w&b

In [52]:
def _init_wandb(cfg: Config, run_name: str, tags: list):
    return wandb.init(
        project="audio-classifier",
        entity="bau-mat-ucl",
        name=run_name,
        tags=tags,
        settings=wandb.Settings(init_timeout=300),
        config=dict(
            sample_rate=cfg.SAMPLE_RATE,
            n_mel=cfg.N_MEL,
            n_fft=cfg.N_FFT,
            hop_length=cfg.HOP_LENGTH,
            duration_ms=cfg.DURATION_MS,
            dropout=cfg.DROPOUT_RATE,
            l2_reg=cfg.L2_REG,
            label_smoothing=cfg.LABEL_SMOOTHING,
            mixup_alpha=cfg.MIXUP_ALPHA,
            batch_size=cfg.BATCH_SIZE,
            epochs=cfg.EPOCHS,
            learning_rate=cfg.LEARNING_RATE,
            early_stop_patience=cfg.EARLY_STOPPING_PATIENCE,
            cosine_restart_epochs=cfg.COSINE_RESTART_EPOCHS,
            cosine_m_mul=cfg.COSINE_M_MUL,
            real_aug_passes=cfg.REAL_AUG_PASSES,
            synth_aug_passes=cfg.SYNTH_AUG_PASSES,
        ),
    )

## Main Pipeline

In [53]:
def run_pipeline(synth_dataset, cfg: Config,
                 run_name: str = None, tags: list = None) -> tuple:
    os.makedirs(cfg.MODEL_DIR, exist_ok=True)
 
    run_name = run_name or f"resnet_{time.strftime('%m%d_%H%M')}"
    run = _init_wandb(cfg, run_name, tags or [])
 
    extractor = MelExtractor(cfg)
    print(f"\n  Input shape : {extractor.input_shape}")
 
    X_train, X_val, X_test, y_train, y_val, y_test, classnames = \
        prepare_dataset(synth_dataset, cfg, extractor)
 
    n_classes = len(classnames)
    model = build_model(extractor.input_shape, n_classes, cfg)
    wandb.log({"n_params": model.count_params()})
 
    history = train_model(model, X_train, y_train, X_val, y_val, cfg, n_classes)
    plot_training_history(history, cfg)
 
    best_model = keras.models.load_model(os.path.join(cfg.MODEL_DIR, "best_model.keras"))
 
    metrics = evaluate_model(
        best_model, X_train, y_train, X_val, y_val,
        X_test, y_test, classnames, cfg,
    )
 
    wandb.log({
        "final/train_acc": metrics["train"],
        "final/val_acc":   metrics["val"],
        "final/test_acc":  metrics["test"],
    })
    artifact = wandb.Artifact("best_model", type="model")
    artifact.add_file(os.path.join(cfg.MODEL_DIR, "best_model.keras"))
    run.log_artifact(artifact)
    wandb.finish()
 
    save_model_and_metadata(best_model, classnames, extractor, metrics, cfg)
 
    print(f"\n  Test accuracy (TTA x{cfg.TTA_STEPS}) : {100*metrics['test']:.2f}%")
    return best_model, metrics, classnames, extractor

In [54]:
dataset = Dataset()
for cls in ("background", "birds", "handsaw", "helicopter"):
    dataset.remove_class(cls)
 
best_model, metrics, classnames, extractor = run_pipeline(
    dataset,
    config,
    run_name=f"resnet_lr{config.LEARNING_RATE}_bs{config.BATCH_SIZE}_{time.strftime('%m%d_%H%M')}",
    tags=["mateo", "macstudio", "metal-gpu", "cosine-decay"],
)


  Input shape : (64, 87, 1)

  DATASET PREPARATION

  [1/4] Real recordings  (../mcu/hands_on_audio_acquisition/audio_files)
        Subfolders : ['training']
----------------------------------------------------------------------
    chainsaw          90 files
    fire              97 files
    fireworks         95 files
    gunshot           97 files


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x108889d20>> (for post_run_cell), with arguments args (<ExecutionResult object at 661483f70, execution_count=54 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 661481ed0, raw_cell="dataset = Dataset()
for cls in ("background", "bir.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/marcbauvir/Mat%C3%A9o/Cours/LELEC210X-project/classification/tryingClassifierMacSetup.ipynb#X41sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost